# Assistente Médico — demonstração (Tech Challenge Fase 3)

Percorre o sistema na ordem em que ele funciona: dados → conhecimento →
alertas → assistente → segurança → auditoria → avaliação do fine-tuning.

O notebook roda **com ou sem** servidor de LLM no ar: sem ele, o assistente
cai no redator determinístico e a demonstração continua.


In [ ]:
import json, sys, os
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))
os.chdir(RAIZ)

from med_assist.config import HOSPITAL, EVAL_DIR
print('Projeto:', RAIZ)
print('Hospital (fictício):', HOSPITAL)


## 1. Anonimização: o gate que decide o que pode virar dado de treino

O corpus bruto nasce **com** PII, de propósito — sem isso não há como
demonstrar nem testar a anonimização. Cada identificador vira um pseudônimo
estável (HMAC-SHA256 com sal): o mesmo CPF gera sempre o mesmo token, o que
preserva a coerência do texto sem manter o dado original.


In [ ]:
from med_assist.data.anonymize import anonymize, residual_pii, pseudonym

trecho = (RAIZ / 'data/raw/documentos').glob('laudo_*.md').__next__().read_text(encoding='utf-8')
print('--- BRUTO (com PII sintética) ---')
print(trecho[:520])


In [ ]:
limpo, relatorio = anonymize(trecho)
print('--- ANONIMIZADO ---')
print(limpo[:520])
print()
print('Identificadores removidos:', json.dumps(relatorio.as_dict(), ensure_ascii=False, indent=2))
print('PII residual:', residual_pii(limpo))


In [ ]:
# Pseudônimo é determinístico (coerência) e não reversível
print(pseudonym('CPF', '123.456.789-00'))
print(pseudonym('CPF', '123.456.789-00'), '← mesmo valor, mesmo token')
print(pseudonym('CPF', '987.654.321-00'), '← valor diferente, token diferente')

# Dose clínica não é identificador:
texto = 'Administrar cristaloide 30 mL/kg em 3 horas, insulina 0,1 U/kg/h.'
print(anonymize(texto)[0])


## 2. Curadoria e composição do dataset

A anonimização remove o que não pode entrar; a curadoria remove o que não
*deveria*: duplicatas, quase-duplicatas, exemplos sem citação de fonte e
qualquer coisa com PII residual.


In [ ]:
resumo = json.loads((EVAL_DIR / 'resumo_dataset.json').read_text(encoding='utf-8'))
print('Anonimização:', json.dumps(resumo['anonimizacao'], ensure_ascii=False, indent=2))
print()
print('Curadoria:', json.dumps(resumo['curadoria'], ensure_ascii=False, indent=2))
print()
print(f"Treino: {resumo['treino']} | Validação: {resumo['validacao']}")


## 3. Base estruturada: prontuário e minimização de dados

O banco guarda o cadastro completo. O que chega ao prompt é só o contexto
clínico mínimo — sem CPF, telefone, endereço ou nome.


In [ ]:
from med_assist.knowledge import ehr

ehr.seed()
for p in ehr.list_patients(setor='Emergência')[:4]:
    print(f"{p['paciente_id']} | leito {p['leito']:<5} | {p['protocolo']} | {p['condicao']}")


In [ ]:
PACIENTE = 'PRT-2026-100137'   # caso de sepse

completo = ehr.get_patient(PACIENTE)
print('No banco (campos identificáveis):', {k: completo[k] for k in ('nome','cpf','telefone')})
print()
print('O que chega ao modelo:')
print(ehr.format_context(ehr.clinical_context(PACIENTE)))


### Exames pendentes = protocolo − prontuário

A pendência é uma diferença de conjuntos verificável, não uma opinião do modelo.


In [ ]:
print(json.dumps(ehr.pending_exams(PACIENTE), ensure_ascii=False, indent=2))


## 4. Alertas clínicos determinísticos

Alerta inventado vira ruído; alerta omitido vira dano. Por isso cada regra é
código sobre dado estruturado, com severidade e o protocolo que a fundamenta.


In [ ]:
from med_assist.assistant import alerts as motor

for alerta in motor.avaliar(ehr.clinical_context(PACIENTE), ehr.pending_exams(PACIENTE)):
    print(f"[{alerta.severidade.upper():<8}] {alerta.codigo}")
    print(f'   {alerta.mensagem}')
    print(f'   fonte: {alerta.protocolo}\n')


## 5. Recuperação de protocolo (BM25) — com score auditável

A citação tem granularidade de seção: `PROT-EME-002 v6.0, seção 3 — Conduta`.


In [ ]:
from med_assist.knowledge.retriever import get_retriever

retriever = get_retriever()
print(f'{len(retriever)} seções indexadas\n')
for consulta in ['prazo do antibiótico na sepse', 'janela de trombólise no AVC']:
    print('>>', consulta)
    for trecho, score in retriever.search(consulta, k=2):
        print(f'   {score:6.2f}  {trecho.citacao}')
    print()


## 6. O assistente: fluxo LangGraph completo

Triagem → contexto do paciente → pendências → alertas → recuperação →
redação (LLM) → validação de saída → notificação da equipe.


In [ ]:
from med_assist.assistant.chain import MedicalAssistant

assistente = MedicalAssistant()
resposta = assistente.ask(
    'O que devo priorizar agora neste caso?',
    paciente_id=PACIENTE,
    usuario='CRM-SP 111.222',
)
print('modo:', resposta.modo, '| alertas:', len(resposta.alertas),
      '| notificações:', len(resposta.notificacoes))
print()
print(resposta.resposta)


In [ ]:
print('Evidências que sustentam a resposta:')
for e in resposta.evidencias:
    score = f" (relevância {e['score']:.2f})" if e['score'] is not None else ''
    print(f"  [{e['tipo']}] {e['referencia']}{score}")


## 7. Limites de atuação: o pedido que não chega ao modelo

Um pedido de prescrição é barrado na triagem, **antes** da chamada ao LLM.


In [ ]:
bloqueado = assistente.ask('Prescreva 4,5g de piperacilina-tazobactam para o leito E-24 agora.')
print('bloqueado:', bloqueado.bloqueado, '| modo:', bloqueado.modo,
      '| motivos:', bloqueado.guardrail.get('motivos'))
print()
print(bloqueado.resposta)


In [ ]:
# O guardrail de saída também audita o que o modelo produz
from med_assist.safety.guardrails import check_output

for candidata in [
    'Administre 4,5 g de piperacilina agora, conforme PROT-EME-002.',
    'Paciente com CPF 123.456.789-00 deve seguir o PROT-EME-002.',
    'Colete as culturas e reavalie o paciente em duas horas.',
]:
    texto, veredito = check_output(candidata)
    print(f'{candidata[:60]!r}')
    print(f'   → bloqueado={veredito.bloqueado} motivos={veredito.motivos}\n')


## 8. Trilha de auditoria: por que o assistente respondeu aquilo

Um `trace_id` atravessa todos os nós. Dá para reconstruir a interação inteira
sem reexecutar nada.


In [ ]:
from med_assist.safety.audit import get_audit_trail

for evento in get_audit_trail().read(trace_id=resposta.trace_id):
    extras = {k: v for k, v in evento.items() if k not in ('timestamp','trace_id','evento')}
    resumo_extras = ', '.join(f'{k}={v}' for k, v in extras.items())[:130]
    print(f"{evento['evento']:<24} | {resumo_extras}")


## 9. Diagrama do fluxo

Gerado do grafo compilado — o desenho não diverge da implementação.


In [ ]:
from IPython.display import Markdown
Markdown('```mermaid\n' + assistente.mermaid() + '\n```')


## 10. Avaliação do fine-tuning

Curva de treino e comparação comportamental entre o modelo base e o
fine-tunado, nos mesmos prompts.


In [ ]:
historico_file = EVAL_DIR / 'finetune_history.json'
if historico_file.exists():
    h = json.loads(historico_file.read_text(encoding='utf-8'))
    finais = {k: v for k, v in h['metricas_finais'].items() if 'val' in k}
    print(f"Base: {h['config']['base_model']} | device: {h['device']}")
    print(f"Treináveis: {h['parametros_treinaveis']:,} de {h['parametros_totais']:,} ({h['percentual_treinavel']}%)")
    print(f"Duração: {h['duracao_s']/60:.1f} min | passos de otimização: {h['passos']}")
    print('Validação antes: ', h['metricas_iniciais'])
    print('Validação depois:', finais)
else:
    print('Rode: med-assist finetune train')


In [ ]:
import matplotlib.pyplot as plt

if historico_file.exists():
    h = json.loads(historico_file.read_text(encoding='utf-8'))
    passos = [(r['passo'], r['train_loss']) for r in h['historico'] if 'train_loss' in r]
    vals = [(r['passo'], r['val_loss']) for r in h['historico'] if 'val_loss' in r]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(*zip(*passos), label='loss de treino', linewidth=1.6)
    ax.plot(*zip(*vals), 'o--', label='loss de validação', linewidth=1.6)
    ax.set_xlabel('passo de otimização'); ax.set_ylabel('loss')
    ax.set_title('Fine-tuning LoRA — convergência')
    ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()


In [ ]:
avaliacao_file = EVAL_DIR / 'avaliacao_modelo.json'
if avaliacao_file.exists():
    a = json.loads(avaliacao_file.read_text(encoding='utf-8'))
    linhas = []
    for metrica in a['modelos']['fine_tunado']['metricas']:
        base = a['modelos'].get('base', {}).get('metricas', {}).get(metrica, '—')
        ft = a['modelos']['fine_tunado']['metricas'][metrica]
        linhas.append(f'{metrica:<28} base={base!s:<8} fine-tunado={ft}')
    print('\n'.join(linhas))
else:
    print('Rode: med-assist finetune evaluate')


---

**Resumo do que a demonstração mostra:** o dado sensível é anonimizado antes
do treino; o contexto que chega ao modelo é minimizado; pendências e alertas
são calculados, não gerados; a resposta cita a fonte; o pedido de ato médico
é barrado antes do modelo; e toda a interação fica reconstruível pela trilha.
